# Quadrature formula

In [ ]:
#    APM41012EP course notebook - Chapter 3 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Quadrature formula
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import numpy as np
import plotly.graph_objs as go
from scipy.integrate import newton_cotes
import plotly.io as pio
pio.templates.default = "seaborn"
import warnings
warnings.filterwarnings('ignore')

We seek to approximate the integral:

$$
I^{[a,b]}(f) = \int_a^b f(x){\mathrm d}x.
$$

In [ ]:
def lagrange(k, xk, x):
    lag = np.ones(x.size)
    for j in range(xk.size):
        if (j!=k): lag *= (x-xk[j])/(xk[k]-xk[j])
    return lag

def lagrange_interpolate(xk, yk, x):
    p = np.zeros(x.size)
    for j in range(xk.size):
        p += yk[j] * lagrange(j, xk, x)
    return p

def lagrange_interpor(xi, yi):
    return lambda x: lagrange_interpolate(xi, yi, x)

## Simple formula

Starting from a quadrature formula with $n+1$ points defined on a reference interval, for example $[−1, 1]$:

$$
I_{n}^{[−1, 1]}(f) = \sum_{k=0}^{n} f(x_k) \omega_k
$$

one can obtain a quadrature formula with $n+1$ points directly on the interval $[a,b]$:

$$
I_{n}^{[a, b]}(f) = \sum_{k=0}^{n} f(\tilde{x}_k) \tilde{\omega}_k,  \quad \tilde{x}_k = a + \frac{b-a}{2} (x_k+1), \quad \tilde{x}_k =  \frac{b-a}{2} \omega_k
$$

where the $n$ quadrature abscissas are the $\tilde{x}_k$ and the associated weights are the $\tilde{x}_k$.

**Example for $f(x) = \cos(\frac{\pi}{2}x)$ on the interval $[0,1]$ with the Newton-Cotes formulas**

In [ ]:
def f(x):
    return np.cos((np.pi/2)*x)

In [ ]:
xplot = np.linspace(0,1,100)

fig = go.Figure(layout_title='Simple formula')
fig.add_trace(go.Scatter(x=xplot, y=f(xplot), name='f(x)', line_width=3, line_color='rgb(76,114,176)'))

n = np.arange(1,20)

for ni in n:
    xk = np.linspace(0, 1, ni+1)
    p = lagrange_interpor(xk, f(xk))
    fig.add_trace(go.Scatter(visible=False, x=xk, y=f(xk), mode='markers', line_color='rgb(221,132,82)', marker_size=8, showlegend=False))
    fig.add_trace(go.Scatter(visible=False, x=xplot, y=p(xplot), fill='tozeroy', line_color='rgb(221,132,82)', name="Integral of the Lagrange polynomial"))

# Make plot visible for s=2
fig.data[1].visible = True
fig.data[2].visible = True

# Create and add slider
steps = []
for i, ni in enumerate(n):
    step = dict(method="update", label = f" {ni+1}",
                args=[{"visible": [el==0 or el==2*i+1 or el==2*i+2  for el in range(len(fig.data))]}])
    steps.append(step)
sliders = [dict(currentvalue={"prefix": "Number of points: "}, steps=steps)]

fig.update_layout(sliders=sliders, height=500, legend=dict(orientation="h", y=1.1))
fig.show()

**Results for $f(x) = \cos(\frac{\pi}{2}x)$ with the Newton-Cotes formulas**

In [ ]:
xplot = np.linspace(0,1,100)
fig = go.Figure(layout_title="Integral of f(x)")
fig.add_trace(go.Scatter(x=xplot, y=f(xplot), fill='tozeroy', name='Integral of f(x)'))
fig.show()

In [ ]:
res_exa = 2/np.pi

n = np.arange(1, 40, 1)

a = 0.
b = 1.

for ni in n:
    wk, _ = newton_cotes(ni, equal=1)
    xk = np.linspace(a, b, ni+1)
    dx = (b - a) / ni
    quad = dx * np.sum(wk * f(xk))
    err = np.abs(quad - res_exa)/res_exa
    print(f"Number of points: {ni+1:2d} --> relative error  : {err:20.15e}")

**Results for $g(x) = \sqrt(x) \log(x)$ with the Newton-Cotes formulas**

In [ ]:
def g(x):
    return  np.sqrt(x)*np.log(x)

In [ ]:
xplot = np.linspace(1.e-20,1,1000)
fig = go.Figure(layout_title="Integral of g(x)")
fig.add_trace(go.Scatter(x=xplot, y=g(xplot), fill='tozeroy', name='Integral of f(x)'))
fig.show()

In [ ]:
res_exa = -4./9.

n = np.arange(1, 40, 1)

a = 1.e-20
b = 1.

for ni in n:
    wk, _ = newton_cotes(ni, equal=1)
    xk = np.linspace(a, b, ni+1)
    dx = (b - a) / ni
    quad = dx * np.sum(wk * g(xk))
    err = np.abs(quad - res_exa)/np.abs(res_exa)
    print(f"Number of points: {ni+1:2d} --> relative error  : {err:20.15e}")

## Composite formulas

For the composite formula, we subdivide the interval $[a,b]$ into $m$ sub-intervals $[t_j,t_{j+1}]$, $j\in\{0, m-1\}$, $a=t_0 < t_j \ldots, <t_m=b$, on which we will use a simple formula:

$$
I_{m,n}^{[a, b]}(f) = \sum_{j=0}^{m-1} I_{m,n}^{[t_j,t_{j+1}]}(f)
$$

**Example for $f(x) = cos(\frac{\pi}{2}x)$ with $m=4$ and the Newton-Cotes formulas**

In [ ]:
xplot = np.linspace(0,1,100)

fig = go.Figure(layout_title='Composite formula with Newton-Cotes')
fig.add_trace(go.Scatter(x=xplot, y=f(xplot), name='f(x)', line_width=3, line_color='rgb(76,114,176)'))

n = np.arange(1,20)

m = 4
x = np.linspace(0, 1, m+1)

for ni in n: 
    for j in range(m):
        xk = np.linspace(x[j], x[j+1], ni+1)
        p = lagrange_interpor(xk, f(xk))
        xplot = np.linspace(x[j],x[j+1],100)
        fig.add_trace(go.Scatter(visible=False, x=xk, y=f(xk), mode='markers', line_color='rgb(76,114,176)', marker_size=8, showlegend=False))
        fig.add_trace(go.Scatter(visible=False, x=xplot, y=p(xplot), fill='tozeroy', name=f"Ij={j}"))

# Make plot visible for n=1
for j in range(m):
    for iplot in range(2):
        #print(iplot+j*2+1)
        fig.data[iplot+j*2+1].visible = True
        
# Create and add slider
steps = []
for i, ni in enumerate(n):
    visible = [el==0 for el in range(len(fig.data))]
    for j in range(m):
        for iplot in range(2):
            visible[ i*(2*m) + (iplot+j*2+1)] = True 
    #step = dict(method="update", label = f" {ni+1}",
    #            args=[{"visible": [el==0 or el==2*m*i+1 or el==2*m*i+2 or el==2*m*i+3 or el==2*m*i+4 for el in range(len(fig.data))]}])
    step = dict(method="update", label = f" {ni+1}",
                args=[{"visible": visible}])
    steps.append(step)
sliders = [dict(currentvalue={"prefix": "Number of points: "}, steps=steps)]

fig.update_layout(sliders=sliders, height=500, legend=dict(orientation="h", y=1.1))

fig.show()

**Results for $f(x) = cos(\frac{\pi}{2}x)$ with the Newton-Cotes formulas**

In [ ]:
res_exa = 2/np.pi

s = 4
w, _ = newton_cotes(s-1, equal=1)
w = w/(s-1)
r = np.linspace(0, 1, s)
print(f"Number of points per interval: {s}\n")

a = 0
b = 1

n = np.array((1, 10, 100, 500, 1000, 10000, 100000))

for i, ni in enumerate(n):
    x = np.linspace(a, b, ni+1)
    res = 0.
    for j in range(ni):
        h = x[j+1]-x[j]
        res +=  h * np.sum(w * f(x[j]+h*r))
    err = np.abs(res - res_exa)/res_exa
    print(f"Number of intervals: {ni:6d} --> relative error  : {err:20.15e}")

**Results for $g(x) = \sqrt(x) \log(x)$ with the Newton-Cotes formulas**

In [ ]:
res_exa = -4./9.

s = 7
w, _ = newton_cotes(s-1, equal=1)
w = w/(s-1)
r = np.linspace(0, 1, s)
print(f"Number of points per interval: {s}\n")

a = 1e-20
b = 1

n = np.array((1, 10, 100, 500, 1000, 10000, 100000))

for i, ni in enumerate(n):
    x = np.linspace(a, b, ni+1)
    res = 0.
    for j in range(ni):
        h = x[j+1]-x[j]
        res +=  h * np.sum(w * g(x[j]+h*r))
    err = np.abs(res - res_exa)/res_exa
    print(f"Number of intervals: {ni:6d} --> relative error  : {err:20.15e}")